# Step 02 — Align paired FASTQs to T2T-CHM13

Companion notebook for step 02 in `workflow/Snakefile`:

- `align_library` aligns **one FASTQ pair** (one library) per Slurm job.
- `alignment_manifest` lists every BAM for later steps
  (`workflow/scripts/02_write_alignment_manifest.py`).

It walks through the step using your real data:

1. The samplesheet: what one row means
2. Where each row comes from in the ENCODE FASTQ manifest
3. The FASTQ files and how read 1 and read 2 line up
4. Library IDs: how each row becomes one Snakemake job
5. The Bowtie2 index files
6. The exact shell command `align_library` runs (from a Snakemake dry-run)
7. A small test alignment you can run and inspect (SAM, FLAG, MAPQ, BAM)
8. The real outputs, once the step has run

Run the `download_fastqs` step before this notebook.

Sections 1–6 only read a few lines of each file (section 6 runs a dry-run,
which starts no jobs) and are fine anywhere. Sections 7 and 8 run
bowtie2/samtools and **must run inside a Slurm job**, not on the login node.
Start one, then launch Jupyter inside it:

```bash
srun --partition=cpu --cpus-per-task=4 --mem=16G --time=01:00:00 --pty bash
```

In [ ]:
import gzip
import os
import shlex
import subprocess
import sys
from pathlib import Path

import pandas as pd
import yaml


def find_repo_root(marker="pixi.toml"):
    # Jupyter starts a notebook's kernel with the notebook's own directory as
    # cwd, not wherever the server was launched from, so paths below can't
    # just assume cwd == repo root.
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / marker).exists():
            return candidate
    raise FileNotFoundError(f"Could not find {marker} above {Path.cwd()}")


REPO_ROOT = find_repo_root()
pd.set_option("display.max_colwidth", None)

In [ ]:
config = yaml.safe_load((REPO_ROOT / "config/config.yaml").read_text())

# Same names and paths as the top of workflow/Snakefile. Config paths may be
# relative (to the repo root) or absolute; `REPO_ROOT / path` handles both.
DATA = REPO_ROOT / config["paths"]["data"]
RESULTS = REPO_ROOT / config["paths"]["results"]
LOGS = REPO_ROOT / config["paths"]["logs"]
BENCHMARKS = REPO_ROOT / config["paths"]["benchmarks"]
REFERENCE_NAME = config["reference"]["name"]
REFERENCE_FASTA = REPO_ROOT / config["reference"]["fasta"]
REFERENCE_INDEX_PREFIX = str(REPO_ROOT / config["reference"]["bowtie2_index_prefix"])
MAX_FRAGMENT_LENGTH = int(config["alignment"]["max_fragment_length"])
ENVMODULES = config["alignment"]["envmodules"]

FASTQ_MANIFEST = DATA / "fastq/manifest.tsv"
FASTQ_SAMPLESHEET = DATA / "fastq/snakemake_tf_chip-seq_samplesheet.csv"
ALIGNMENT_DIR = RESULTS / "alignment" / REFERENCE_NAME
ALIGNMENT_MANIFEST = ALIGNMENT_DIR / "alignment_manifest.tsv"

print("reference:     ", REFERENCE_NAME)
print("index prefix:  ", REFERENCE_INDEX_PREFIX)
print("samplesheet:   ", FASTQ_SAMPLESHEET)
print("BAMs go to:    ", ALIGNMENT_DIR)
print("bowtie2 -X:    ", MAX_FRAGMENT_LENGTH)
print("modules:       ", " ".join(ENVMODULES))

## 1. The samplesheet

ENCODEfetch writes the samplesheet during step 01, and the Snakefile builds
one alignment job per row of it. Despite the `.csv` name, its columns are
separated by **tabs**. Printing the raw header line with `repr` makes the `\t`
characters visible; that is why the Snakefile's `libraries()` reads it with
`delimiter="\t"`.

In [ ]:
with FASTQ_SAMPLESHEET.open() as handle:
    print(repr(handle.readline()))

Read with the right separator, it is a table with **one row per library**:
one pair of FASTQ files (read 1 + read 2) from one sequencing run. Paths are
shortened here to fit.

In [ ]:
samplesheet = pd.read_csv(FASTQ_SAMPLESHEET, sep="\t", dtype=str, keep_default_na=False)
fastq_root = DATA / "fastq/files"

view = samplesheet.copy()
for column in ("fastq_1", "fastq_2"):
    view[column] = view[column].map(lambda path: os.path.relpath(path, fastq_root))
view

| Column | Meaning |
|---|---|
| `sample` | ENCODE experiment accession (`ENCSR…`) |
| `group` | `case` = the ChIP pull-down with the antibody; `control` = the matched input DNA (no antibody) |
| `replicate` | Biological replicate number within that experiment |
| `fastq_1` / `fastq_2` | Read 1 and read 2 FASTQ of one paired-end run (`ENCFF…` file accessions) |
| `control` | For a case row, the control experiment it will be compared against in peak calling; empty for control rows |
| `control_replicate` | Which replicate of that control to use |
| `antibody` | ChIP target |

Alignment only needs `fastq_1` and `fastq_2`. The other columns are copied
into the alignment manifest so a later peak-calling step can pair each case
BAM with its control.

Counting rows per replicate shows why there are more rows than replicates:

In [ ]:
(
    samplesheet.groupby(["sample", "group", "replicate"])
    .size()
    .rename("libraries (FASTQ pairs)")
    .reset_index()
)

## 2. Where each row comes from: the FASTQ manifest

`manifest.tsv` (also from step 01) has the full ENCODE metadata, one row per
FASTQ pair. A few of its columns explain the samplesheet:

- `biological_replicates` / `technical_replicates`: `1_1` means biological
  replicate 1, technical replicate 1. Rows sharing the same value come from
  the same library, sequenced in more than one run.
- `file_accession` / `file_accession_r2`: the read 1 and read 2 FASTQs.
- `md5sum`: the checksum step 01 verified the download against.

In [ ]:
manifest = pd.read_csv(FASTQ_MANIFEST, sep="\t", dtype=str, keep_default_na=False)
print(manifest.shape[1], "columns in the manifest; a selection:")
manifest[
    [
        "experiment_accession",
        "is_control",
        "target_label",
        "biosample_term_name",
        "biological_replicates",
        "technical_replicates",
        "file_accession",
        "file_accession_r2",
        "run_type",
        "platform",
        "md5sum",
    ]
]

Joining on the FASTQ path confirms every samplesheet row is one manifest row:

In [ ]:
samplesheet.merge(
    manifest[["local_path", "technical_replicates", "file_accession", "file_accession_r2"]],
    left_on="fastq_1",
    right_on="local_path",
    how="left",
)[["sample", "group", "replicate", "technical_replicates", "file_accession", "file_accession_r2"]]

## 3. The FASTQ files

A FASTQ file stores reads as 4-line records:

```
@read name
bases (A/C/G/T, N = base the sequencer couldn't call)
+
one quality character per base (# = very low, letters = high)
```

In paired-end sequencing each DNA fragment is read from both ends. Read 1 of
every fragment is in `fastq_1` and read 2 is in `fastq_2`, **in the same
order**. That order is how bowtie2 knows which two reads belong together.

In [ ]:
pd.DataFrame(
    [
        {
            "sample": row["sample"],
            "replicate": row["replicate"],
            "read": read,
            "file": Path(row[column]).name,
            "exists": Path(row[column]).exists(),
            "size_GB": round(Path(row[column]).stat().st_size / 1e9, 2),
        }
        for _, row in samplesheet.iterrows()
        for read, column in ((1, "fastq_1"), (2, "fastq_2"))
    ]
)

In [ ]:
def read_fastq_records(path, n):
    """Return the first n records of a gzipped FASTQ as (name, sequence, quality)."""
    records = []
    with gzip.open(path, "rt") as handle:
        for _ in range(n):
            name = handle.readline().rstrip()
            if not name:
                break
            sequence = handle.readline().rstrip()
            handle.readline()  # the "+" separator line
            quality = handle.readline().rstrip()
            records.append((name, sequence, quality))
    return records


example = samplesheet.iloc[0]
for column in ("fastq_1", "fastq_2"):
    print(f"--- {column}: {Path(example[column]).name}")
    for name, sequence, quality in read_fastq_records(example[column], 2):
        print(name, sequence, "+", quality, sep="\n")

The two files' read names are identical up to the space; after it, `1:…` or
`2:…` says which end was read. The cell below checks that the first 1,000
names match in every library. It also pulls the flowcell and lane out of the
Illumina read name (`instrument:run:flowcell:lane:tile:x:y`), which shows
whether two libraries of the same replicate came from different runs.

In [ ]:
N_CHECK = 1000
checks = []
for _, row in samplesheet.iterrows():
    r1 = read_fastq_records(row["fastq_1"], N_CHECK)
    r2 = read_fastq_records(row["fastq_2"], N_CHECK)
    names_match = len(r1) == len(r2) and all(
        a[0].split()[0] == b[0].split()[0] for a, b in zip(r1, r2)
    )
    fields = r1[0][0][1:].split()[0].split(":")
    checks.append(
        {
            "sample": row["sample"],
            "replicate": row["replicate"],
            "fastq_1": Path(row["fastq_1"]).name,
            "fastq_2": Path(row["fastq_2"]).name,
            "first_1000_names_match": names_match,
            "read_length": len(r1[0][1]),
            "flowcell": fields[2] if len(fields) >= 4 else "",
            "lane": fields[3] if len(fields) >= 4 else "",
        }
    )
pd.DataFrame(checks)

## 4. Library IDs: one row, one job

`library_id()` in the Snakefile names each library
`sample__group__repN__R1accession_R2accession`. Including the FASTQ
accessions keeps the name unique when one replicate has several FASTQ pairs.

Snakemake works backwards from the files it is asked for. `alignment_manifest`
asks for one BAM per library ID; each BAM path matches the output pattern
`…/{library}.sorted.bam` of `align_library`, so Snakemake creates one
`align_library` job per library, with the `{library}` wildcard set to that
ID. The same ID names the job's logs and benchmark file.

In [ ]:
def library_id(row):
    # Same as library_id() in workflow/Snakefile.
    r1_accession = Path(row["fastq_1"]).name.split(".")[0]
    r2_accession = Path(row["fastq_2"]).name.split(".")[0]
    return (
        f"{row['sample']}__{row['group']}__rep{row['replicate']}"
        f"__{r1_accession}_{r2_accession}"
    )


libraries = samplesheet.assign(library_id=samplesheet.apply(library_id, axis=1))
libraries["bam"] = libraries["library_id"].map(lambda i: str(ALIGNMENT_DIR / f"{i}.sorted.bam"))
libraries["bowtie2_log"] = libraries["library_id"].map(lambda i: str(LOGS / "alignment" / f"{i}.bowtie2.log"))
libraries["benchmark"] = libraries["library_id"].map(lambda i: str(BENCHMARKS / "alignment" / f"{i}.tsv"))

print("output folder:", ALIGNMENT_DIR)
libraries[["library_id"]].assign(
    bam=libraries["bam"].map(lambda p: Path(p).name),
    bowtie2_log=libraries["bowtie2_log"].map(lambda p: Path(p).name),
)

## 5. The Bowtie2 index

Searching a 3-billion-base FASTA directly for every one of tens of millions of
reads would be far too slow. `bowtie2-build` converts the genome once into an
**FM-index**, a compressed structure that finds where a short string occurs
in time proportional to the string's length, not the genome's. After that,
bowtie2 only needs these 6 files:

| File | What's in it |
|---|---|
| `.1.bt2` | The Burrows–Wheeler transform (BWT) of the genome plus lookup tables and the sequence names. This is what bowtie2 searches to find where part of a read matches. |
| `.2.bt2` | A sample of the suffix array. A match found in the BWT is only a position in that transformed text; this converts it to a chromosome coordinate. |
| `.3.bt2` | The layout of the sequence: where each chromosome starts and ends, and any runs of `N`. Tiny. |
| `.4.bt2` | The genome sequence itself, packed at 2 bits per base (4 bases per byte). Used to score the whole read against the genome once a candidate position is found. |
| `.rev.1.bt2` | Same as `.1`, built from the genome **written backwards** (reversed, not reverse-complemented). |
| `.rev.2.bt2` | Same as `.2`, for the reversed genome. |

**Why the reversed copy?** A BWT search extends a match one base at a time in
only one direction. A second index of the reversed genome lets bowtie2 extend
matches in the other direction too, so it can allow a mismatch in either half
of a seed without backtracking through huge numbers of possibilities. There
are no `rev.3`/`rev.4` files because the sequence only needs to be stored
once. Reads from the opposite DNA strand are found by also searching each
read's reverse complement, not by indexing the other strand.

**`.bt2` vs `.bt2l`:** `.bt2l` files hold the same data with 64-bit
positions, needed for genomes over about 4 billion bases. T2T-CHM13 (~3.1
billion) fits in `.bt2`. `bowtie2_index_files()` in the Snakefile checks
which kind exists.

In [ ]:
index_prefix = Path(REFERENCE_INDEX_PREFIX)
index_files = sorted(index_prefix.parent.glob(f"{index_prefix.name}.*.bt2*"))
pd.DataFrame(
    {
        "file": [f.name for f in index_files],
        "size_MB": [round(f.stat().st_size / 1e6, 1) for f in index_files],
    }
)

A quick check of the `.4.bt2` description: at 4 bases per byte, its size
should be the genome length divided by 4 (rounded up to a whole byte). The
genome length comes from the FASTA index (`.fai`), which lists every sequence
and its length.

In [ ]:
fai = pd.read_csv(f"{REFERENCE_FASTA}.fai", sep="\t", header=None, usecols=[0, 1], names=["sequence", "length"])
genome_bases = int(fai["length"].sum())
four = next(f for f in index_files if ".rev." not in f.name and f.name.endswith((".4.bt2", ".4.bt2l")))

print(f"{len(fai)} sequences, {genome_bases:,} bases in total")
print(f"genome bases / 4 = {genome_bases / 4:>15,.0f} bytes")
print(f"{four.name} size = {four.stat().st_size:>15,} bytes")

## 6. The command `align_library` runs

The cell below asks Snakemake itself for the command of the first library:
a dry-run (`--dry-run`) of just that BAM, forced (`--force`) so it is shown
even if the BAM already exists. Nothing is submitted.

Before the command, Snakemake loads the modules listed in
`alignment.envmodules` (`module purge && module load …`) and runs everything
under `set -euo pipefail`, so the job fails if any part fails, including
bowtie2 on the left of the pipe. Snakemake then deletes the incomplete BAM.

What each piece does:

- `tmp_prefix=/scratch/$USER/$SLURM_JOB_ID/<library>`: where `samtools sort`
  puts temporary chunks. Scratch is fast local disk.
- `bowtie2 --threads N -X <max fragment> -x <index prefix> -1 <R1> -2 <R2>`
  aligns every read pair and writes **SAM** (plain text, one line per read)
  to stdout. Its summary goes to `<library>.bowtie2.log`. `-X` is explained
  in 7e.
- `| samtools sort -@ 4 -m 2G -T $tmp_prefix -o <bam> -` reads that SAM from
  stdin (`-`), sorts the reads by chromosome and position (holding up to 2 GB
  per thread in memory, spilling the rest to scratch), and writes compressed
  **BAM** straight into the results folder. Position-sorted BAM is what genome
  browsers, duplicate marking and peak callers expect.
- `samtools index` writes `<bam>.bai`, which lets tools jump straight to a
  region instead of reading the whole file.

In [ ]:
first_bam = libraries["bam"].iloc[0]
dry_run = subprocess.run(
    [sys.executable, "-m", "snakemake", "--profile", "profiles/karakoram",
     "--dry-run", "--nolock", "--force", first_bam],
    cwd=REPO_ROOT,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)
print(dry_run.stdout)

## 7. Try it: align a few thousand read pairs

This runs the same steps on the first `N_PAIRS` read pairs of one library, and
keeps the unsorted SAM so you can read bowtie2's output directly. It uses the
modules and `-X` value from the config, like `align_library`. Loading the T2T
index takes about 4 GB of memory, so the cells refuse to run outside a Slurm
job. Set `RUN_TEST_ALIGNMENT = True` to use this section.

Test files go under `/scratch/$USER/$SLURM_JOB_ID/`, which Slurm cleans up
after the job ends.

In [ ]:
RUN_TEST_ALIGNMENT = False
N_PAIRS = 20_000
TEST_LIBRARY = libraries.iloc[0]
TEST_THREADS = int(os.environ.get("SLURM_CPUS_PER_TASK", "1"))
# Set to 500 (bowtie2's default) and re-run 7b-7e to see what -X changes.
TEST_MAX_FRAGMENT_LENGTH = MAX_FRAGMENT_LENGTH

if RUN_TEST_ALIGNMENT and "SLURM_JOB_ID" not in os.environ:
    raise RuntimeError(
        "Not inside a Slurm job. Start one with srun (see the top of this notebook); "
        "bowtie2 must not run on the login node."
    )

test_dir = (
    Path("/scratch")
    / os.environ.get("USER", "unknown")
    / os.environ.get("SLURM_JOB_ID", "local")
    / "02_align_fastqs_notebook"
)


def run_shell(command):
    """Run a shell command with the config's modules loaded.

    Modules load before `set -euo pipefail`: sourcing /etc/profile on Karakoram
    fails under `set -u`.
    """
    result = subprocess.run(
        [
            "bash",
            "-lc",
            "source /etc/profile >/dev/null 2>&1 || true; "
            f"module load {' '.join(ENVMODULES)}; set -euo pipefail; {command}",
        ],
        capture_output=True,
        text=True,
    )
    if result.returncode != 0:
        raise RuntimeError(f"Command failed ({result.returncode}):\n{command}\n\n{result.stderr}")
    return result


print("test library:", TEST_LIBRARY["library_id"])
print("threads:     ", TEST_THREADS)
print("bowtie2 -X:  ", TEST_MAX_FRAGMENT_LENGTH)
print("test folder: ", test_dir)

### 7a. Take the first read pairs

Taking the first `N_PAIRS` records from **both** files keeps the pairs
together, because the two files are in the same order.

In [ ]:
if RUN_TEST_ALIGNMENT:
    test_dir.mkdir(parents=True, exist_ok=True)
    test_r1 = test_dir / "test_R1.fastq"
    test_r2 = test_dir / "test_R2.fastq"
    for source, target in ((TEST_LIBRARY["fastq_1"], test_r1), (TEST_LIBRARY["fastq_2"], test_r2)):
        with target.open("w") as out:
            for name, sequence, quality in read_fastq_records(source, N_PAIRS):
                out.write(f"{name}\n{sequence}\n+\n{quality}\n")
    print("wrote", test_r1)
    print("wrote", test_r2)

### 7b. Run bowtie2 and read its summary

`align_library` pipes straight into `samtools sort`; here `-S` writes plain
SAM to a file instead. The summary it prints is the same text that ends up in
each library's `.bowtie2.log`:

- **aligned concordantly exactly 1 time**: both reads aligned, facing each
  other at a sensible distance, and only one genome position fits best.
- **aligned concordantly >1 times**: the pair fits equally well in more than
  one place (repeats). These get low MAPQ (see 7e).
- **aligned discordantly**: both reads aligned, but not as a proper pair
  (wrong orientation, too far apart, or different chromosomes). "Too far
  apart" means a fragment longer than `-X`; see 7e.
- **overall alignment rate**: fraction of all reads that aligned somewhere.

The `[WARNING] Failed to launch x86-64-v3 version` lines come from the
cluster's bowtie2 module falling back to its generic build; they are harmless.

In [ ]:
if RUN_TEST_ALIGNMENT:
    test_sam = test_dir / "test.sam"
    result = run_shell(
        f"bowtie2 --threads {TEST_THREADS} -X {TEST_MAX_FRAGMENT_LENGTH} "
        f"-x {shlex.quote(REFERENCE_INDEX_PREFIX)} "
        f"-1 {shlex.quote(str(test_r1))} -2 {shlex.quote(str(test_r2))} "
        f"-S {shlex.quote(str(test_sam))}"
    )
    print(result.stderr)

### 7c. The SAM header

Lines starting with `@` describe the file: `@HD` (format version and sort
order), one `@SQ` per reference sequence with its name (`SN`) and length
(`LN`), and `@PG`, the program and exact command line that produced it.

The shared T2T-CHM13v2.0 bundle names chromosomes by RefSeq accession rather
than `chr1`, `chr2`, …: `NC_060925.1` is chr1 (248,387,328 bp) through
`NC_060946.1` chr22, `NC_060947.1` chrX and `NC_060948.1` chrY. Anything
compared against these BAMs later (peaks, annotations, blacklists) has to use
the same names.

In [ ]:
if RUN_TEST_ALIGNMENT:
    with test_sam.open() as handle:
        header = [line.rstrip() for line in handle if line.startswith("@")]
    print(len(header), "header lines:\n")
    print("\n".join(header[:4] + ["..."] + header[-2:]))

### 7d. The alignment records

Every other line is one read, with 11 fixed columns and then optional tags:

| Column | Meaning |
|---|---|
| `QNAME` | Read name (the same for both reads of a pair) |
| `FLAG` | Bit flags describing the read (decoded below) |
| `RNAME`, `POS` | Chromosome and 1-based leftmost position of the alignment |
| `MAPQ` | Mapping quality: how confident bowtie2 is that this is the right position |
| `CIGAR` | How the read lines up, e.g. `101M` = 101 bases aligned; `I`/`D` = insertion/deletion, `S` = clipped |
| `RNEXT`, `PNEXT` | Where the mate aligned (`=` means same chromosome) |
| `TLEN` | Fragment length implied by the pair (negative on the rightmost read) |
| `SEQ`, `QUAL` | Bases and qualities (reverse-complemented if the read aligned to the minus strand) |

Common bowtie2 tags: `AS:i` alignment score; `XS:i` score of the second-best
position (present = the read also fits elsewhere); `NM:i` number of
mismatches; `YS:i` the mate's score; `YT:Z` the pair type: `CP` concordant,
`DP` discordant, `UP` not aligned as a pair.

In [ ]:
SAM_COLUMNS = ["QNAME", "FLAG", "RNAME", "POS", "MAPQ", "CIGAR", "RNEXT", "PNEXT", "TLEN", "SEQ", "QUAL"]

if RUN_TEST_ALIGNMENT:
    records = []
    with test_sam.open() as handle:
        for line in handle:
            if line.startswith("@"):
                continue
            fields = line.rstrip("\n").split("\t")
            record = dict(zip(SAM_COLUMNS, fields[:11]))
            record["tags"] = " ".join(fields[11:])
            records.append(record)
    sam = pd.DataFrame(records).astype({"FLAG": int, "POS": int, "MAPQ": int, "PNEXT": int, "TLEN": int})
    print(len(sam), "records (2 per read pair)")
    display(sam.head(6).drop(columns=["SEQ", "QUAL"]))

`FLAG` is a sum of powers of two, each meaning one yes/no property. For
example 99 = 1 + 2 + 32 + 64: paired, proper pair, mate on reverse strand,
first in pair.

In [ ]:
FLAG_BITS = {
    1: "paired",
    2: "proper pair",
    4: "read unmapped",
    8: "mate unmapped",
    16: "read on reverse strand",
    32: "mate on reverse strand",
    64: "first in pair (R1)",
    128: "second in pair (R2)",
    256: "secondary",
    512: "fails QC",
    1024: "duplicate",
    2048: "supplementary",
}


def decode_flag(flag):
    return ", ".join(name for bit, name in FLAG_BITS.items() if flag & bit)


if RUN_TEST_ALIGNMENT:
    flag_counts = sam["FLAG"].value_counts().rename_axis("FLAG").reset_index(name="reads")
    flag_counts["meaning"] = flag_counts["FLAG"].map(decode_flag)
    display(flag_counts)

### 7e. MAPQ, chromosomes and fragment length

`MAPQ` 0 or 1 means the read fits equally well at several places, typically
repeats; bowtie2's highest value is 42. `filtering.minimum_mapq` in the config
is meant for the later filtering step; `align_library` keeps every read.

`TLEN` of properly paired reads estimates the length of the DNA fragments that
went into the sequencer. bowtie2 only calls a pair "proper" if its fragment is
at most `-X` long. With bowtie2's default `-X 500`, the maximum below is
exactly 500 and about 30% of this data's pairs were flagged discordant
(FLAG without bit 2, `YT:Z:DP`), almost all simply because their fragment was
longer than 500 bp. A later "keep proper pairs only" filter would throw them
away. That is why the workflow sets `alignment.max_fragment_length: 2000`, the
value ENCODE's own ChIP-seq pipeline uses. The last lines count how many
non-proper pairs are longer than 500 and 2000 bp.

In [ ]:
if RUN_TEST_ALIGNMENT:
    mapped = sam[(sam["FLAG"] & 4) == 0]
    mapq_bins = pd.cut(mapped["MAPQ"], [-1, 0, 1, 9, 29, 42], labels=["0", "1", "2-9", "10-29", "30-42"])
    display(mapq_bins.value_counts(sort=False).rename_axis("MAPQ").rename("mapped reads").to_frame())
    display(mapped["RNAME"].value_counts().rename("mapped reads").to_frame().T)

    proper = sam[((sam["FLAG"] & 2) != 0) & (sam["TLEN"] > 0)]
    print(f"fragment length of proper pairs (-X {TEST_MAX_FRAGMENT_LENGTH}):")
    print(proper["TLEN"].describe().round(1).to_string())

    # One row per pair: both mates mapped, same chromosome, leftmost mate (TLEN > 0).
    same_chrom_pairs = sam[((sam["FLAG"] & 12) == 0) & (sam["RNEXT"] == "=") & (sam["TLEN"] > 0)]
    not_proper = same_chrom_pairs[(same_chrom_pairs["FLAG"] & 2) == 0]
    print(f"\nread pairs in the test:                        {N_PAIRS:>6}")
    print(f"proper pairs:                                   {len(proper):>6}")
    print(f"both mates on one chromosome, not proper pair:  {len(not_proper):>6}")
    print(f"  ...of which fragment longer than 500 bp:      {(not_proper['TLEN'] > 500).sum():>6}")
    print(f"  ...of which fragment longer than 2000 bp:     {(not_proper['TLEN'] > 2000).sum():>6}")

### 7f. Sort, index, and compare with the SAM

These are the remaining steps of `align_library`. The sorted BAM lists reads
by chromosome and position (the SAM followed FASTQ order), and it is much
smaller than the SAM because BAM is compressed binary. `samtools flagstat`
gives the same kind of counts you will want for every real BAM.

In [ ]:
if RUN_TEST_ALIGNMENT:
    test_bam = test_dir / "test.sorted.bam"
    run_shell(
        f"samtools sort -@ {max(1, TEST_THREADS - 1)} "
        f"-o {shlex.quote(str(test_bam))} {shlex.quote(str(test_sam))}"
    )
    run_shell(f"samtools index {shlex.quote(str(test_bam))}")

    sizes = {path.name: path.stat().st_size for path in (test_sam, test_bam, Path(f"{test_bam}.bai"))}
    display(pd.Series(sizes, name="bytes").to_frame())

    print("first 3 records of the sorted BAM:")
    print(run_shell(f"samtools head -h 0 -n 3 {shlex.quote(str(test_bam))}").stdout)
    print(run_shell(f"samtools flagstat {shlex.quote(str(test_bam))}").stdout)

## 8. The real outputs (after step 02 has run)

The alignment manifest has one row per BAM, with the samplesheet columns
carried over. Each library also has its own bowtie2 log (the summary from 7b)
and a benchmark file from Snakemake with the job's wall-clock time (`s`) and
peak memory (`max_rss`, MB), useful for tuning `mem_mb` and `runtime` in
`profiles/karakoram/config.yaml`.

In [ ]:
if ALIGNMENT_MANIFEST.exists():
    display(pd.read_csv(ALIGNMENT_MANIFEST, sep="\t"))
else:
    print("No alignment manifest yet:", ALIGNMENT_MANIFEST)

summary = []
for _, row in libraries.iterrows():
    entry = {"library_id": row["library_id"], "bam_exists": Path(row["bam"]).exists()}
    log = Path(row["bowtie2_log"])
    if log.exists():
        rates = [line.split()[0] for line in log.read_text().splitlines() if "overall alignment rate" in line]
        entry["overall_alignment_rate"] = rates[-1] if rates else "incomplete"
    benchmark = Path(row["benchmark"])
    if benchmark.exists():
        bench = pd.read_csv(benchmark, sep="\t").iloc[0]
        entry["runtime_min"] = round(bench["s"] / 60, 1)
        entry["max_rss_GB"] = round(bench["max_rss"] / 1024, 1)
    summary.append(entry)
pd.DataFrame(summary)

`samtools flagstat` reads a whole BAM, which takes a few minutes per file for
real data, so it is off by default and needs a Slurm job like section 7.

In [ ]:
RUN_FLAGSTAT = False

if RUN_FLAGSTAT:
    if "SLURM_JOB_ID" not in os.environ:
        raise RuntimeError("Not inside a Slurm job; start one with srun first.")
    for bam_path in libraries["bam"]:
        if Path(bam_path).exists():
            print(f"=== {Path(bam_path).name}")
            print(run_shell(f"samtools flagstat -@ {TEST_THREADS} {shlex.quote(bam_path)}").stdout)
        else:
            print(f"=== missing: {bam_path}")